# Module 3 lab: Shor's algorithm and period finding

**Quantum Computing Intermediate · Module 3 · about 60 minutes** · notebook version 2026-10-05

Shor's algorithm factors a number N by finding the **order** r of a number a: the smallest r with aʳ mod N = 1. A classical computer finds the factors from r in a few steps; the hard part, finding r, is done by **phase estimation** (Module 1) on the gate that multiplies by a mod N.

In this lab you factor N = 15:

1. find orders classically, `order(a, N)`, and turn an order into factors, `factors_from_order(a, r, N)`;
2. use a **prepared** controlled multiplication circuit `c_amod15(a, power)`, hard-coded for N = 15;
3. build the quantum order-finding circuit, `order_finding(a, t)`;
4. read r from the measured results with continued fractions, `period_from_counts(counts, t, a, N)`;
5. factor 15 for every allowed a, and get your verification value.

The **Module 3 lab check** in Canvas asks for results from this notebook, so keep it open in a second tab. Run each code cell with **Shift + Enter**, in order.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser.

In [ ]:
import math
from fractions import Fraction
import numpy as np
import qsim
from qsim import QuantumCircuit, Statevector, AerSimulator, plot_histogram

sim = AerSimulator(seed_simulator=7)
print("Ready. qsim", qsim.__version__, "and NumPy", np.__version__)

## Step 1: the order of a mod N

For a number a that shares no factor with N, the powers a¹, a², a³, ... mod N repeat. The **order** r is the smallest r > 0 with aʳ mod N = 1. For example, for a = 2 and N = 15: 2, 4, 8, 16 mod 15 = 1, so r = 4.

**Your task:** write `order(a, N)` by trying r = 1, 2, 3, ... and returning the first r with `pow(a, r, N) == 1`. This classical search takes about r steps, and r can be almost as large as N; for a 2,048-bit N that is hopeless, which is why the quantum part is needed.

The cell after it prints the order of every a that the prepared circuit supports. The lab check asks for the order of 7.

In [ ]:
def order(a, N):
    """The smallest r > 0 with a^r mod N == 1 (classical search)."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete order() first.")

In [ ]:
for a in (2, 4, 7, 8, 11, 13):
    powers = [pow(a, k, 15) for k in range(1, 7)]
    print(f"a = {a:2d}:  a^1 .. a^6 mod 15 = {powers}   order r = {order(a, 15)}")

## Step 2: from the order to the factors

If r is **even**, write h = a^(r/2) mod N. Then h² − 1 = (h − 1)(h + 1) is a multiple of N. If also h ≠ N − 1, neither h − 1 nor h + 1 is a multiple of N, so each shares a factor with N, and the greatest common divisors gcd(h − 1, N) and gcd(h + 1, N) are factors of N. If r is odd, or h = N − 1, this a gives no factor: pick another a.

**Your task:** write `factors_from_order(a, r, N)`. Return the two factors as a sorted tuple, or `None` when r is odd or h = N − 1. Use `math.gcd` and `pow(a, r // 2, N)`.

The cell after it uses a = 7 and r = 4: h = 7² mod 15 = 4, gcd(3, 15) = 3 and gcd(5, 15) = 5. The lab check asks for gcd(7² + 1, 15).

In [ ]:
def factors_from_order(a, r, N):
    """The factors (p, q), sorted, from an even order r with a^(r/2) mod N != N - 1; otherwise None."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete factors_from_order() first.")

In [ ]:
print("factors from a = 7, r = 4:", factors_from_order(7, 4, 15))
print("gcd(7**2 - 1, 15) =", math.gcd(7 ** 2 - 1, 15), "   gcd(7**2 + 1, 15) =", math.gcd(7 ** 2 + 1, 15))
print("a = 14, r = 2 (14 = -1 mod 15):", factors_from_order(14, 2, 15))

## Step 3: the prepared modular multiplication for N = 15

The quantum part needs the gate U that multiplies a 4-qubit register by a mod 15: U|x⟩ = |a·x mod 15⟩. Its eigenphases are s/r for s = 0, 1, ..., r − 1, so phase estimation on U reveals r.

**Read this before you use it.** The function `c_amod15(a, power)` below is **prepared for you and hard-coded for N = 15**: for each allowed a it is a fixed pattern of controlled swaps (`cswap`, new in qsim 1.8.0) and controlled X gates that happens to permute the residues correctly. It is not a general method. A real run of Shor's algorithm on a large N builds modular multiplication from quantum adders, from a and N alone, and that circuit is far larger; it is the main cost of the algorithm.

Run the cell: with the control qubit set to 1, the register value x becomes 7·x mod 15.

In [ ]:
A_VALUES = (2, 4, 7, 8, 11, 13)   # the values of a that c_amod15 supports


def c_amod15(a, power):
    """Prepared: controlled multiplication by a^power mod 15, on qubits [control, x0, x1, x2, x3].
    Hard-coded for N = 15 and a in (2, 4, 7, 8, 11, 13): swaps and X gates that permute the 15 residues."""
    if a not in A_VALUES:
        raise ValueError("c_amod15 works only for a = 2, 4, 7, 8, 11 or 13")
    U = QuantumCircuit(5)
    for _ in range(power):
        if a in (2, 13):
            U.cswap(0, 3, 4); U.cswap(0, 2, 3); U.cswap(0, 1, 2)
        if a in (7, 8):
            U.cswap(0, 1, 2); U.cswap(0, 2, 3); U.cswap(0, 3, 4)
        if a in (4, 11):
            U.cswap(0, 2, 4); U.cswap(0, 1, 3)
        if a in (7, 11, 13):
            for q in range(1, 5):
                U.cx(0, q)
    return U


def qft_dagger(t):
    """Prepared: the inverse QFT on t qubits (the inverse of Module 1's qft(t), gates in reverse order)."""
    qc = QuantumCircuit(t)
    for i in range(t // 2):
        qc.swap(i, t - 1 - i)
    for j in range(t):
        for k in range(j):
            qc.cp(-math.pi / 2 ** (j - k), k, j)
        qc.h(j)
    return qc


qc_test = QuantumCircuit(5)
qc_test.x(0)            # control qubit on
qc_test.x(1)            # register x = 1 (qubit 1 is the least significant bit of x)
for p in range(5):
    state = Statevector(qc_test)
    x = int(np.argmax(np.abs(np.asarray(state)))) >> 1
    print(f"after {p} multiplications by 7: x = {x:2d}    (7^{p} mod 15 = {pow(7, p, 15)})")
    qc_test.compose(c_amod15(7, 1), inplace=True)

## Step 4: the order-finding circuit

This is phase estimation (Module 1) with U = multiplication by a mod 15. The circuit has t **counting qubits** (0 to t − 1) and a 4-qubit **register** (qubits t to t + 3):

1. an X on qubit t, so the register holds |1⟩;
2. an H on every counting qubit;
3. counting qubit k controls U raised to the power 2ᵏ: `qc.compose(c_amod15(a, 2**k), qubits=[k] + list(range(t, t + 4)), inplace=True)`;
4. the inverse QFT on the counting qubits: `qc.compose(qft_dagger(t), qubits=list(range(t)), inplace=True)`;
5. measure counting qubit k into classical bit k.

Why the register starts in |1⟩: |1⟩ is not an eigenvector of U, but it is an equal superposition of r eigenvectors, with eigenphases s/r. Each shot of phase estimation then reads one of them at random: the result m gives m/2ᵗ ≈ s/r for a random s.

**Your task:** write `order_finding(a, t)`.

In [ ]:
def order_finding(a, t):
    """Order finding for N = 15: t counting qubits, a 4-qubit register (qubits t to t + 3) starting in |1>."""
    qc = QuantumCircuit(t + 4, t)
    # YOUR CODE HERE
    raise NotImplementedError("Complete order_finding() first.")
    return qc

## Step 5: run it for a = 7

With 8 counting qubits, 2⁸ = 256 is a multiple of r = 4, so every phase s/4 is exact and only r results appear, each about a quarter of the time. The cell prints each result m, its phase m/256 and the fraction it equals. The lab check asks which four results appear.

In [ ]:
t = 8
counts = sim.run(order_finding(7, t), shots=1000).result().get_counts()
for key in sorted(counts):
    m = int(key, 2)
    print(f"result {key} = m = {m:3d}   shots {counts[key]:4d}   m/256 = {m / 2 ** t:.4f} = {Fraction(m, 2 ** t)}")
plot_histogram(counts, title="order finding for a = 7, 8 counting qubits")

## Step 6: read r with continued fractions

Each result gives a fraction m/2ᵗ ≈ s/r. `Fraction(m, 2**t).limit_denominator(N)` finds the closest fraction with a denominator of at most N, which is s/r in lowest terms. Its denominator is r, **unless s and r share a factor**: for m = 128, 128/256 = 1/2, whose denominator 2 is not the order of 7 (7² mod 15 = 4). So check each candidate with `pow(a, r, N) == 1`, and try the next result if it fails. For a real N with no exact phases, the same method works on the results near s/r.

**Your task:** write `period_from_counts(counts, t, a, N)`: go through the results, most frequent first; for each, take the denominator of `Fraction(int(key, 2), 2**t).limit_denominator(N)`; return the first one with `pow(a, r, N) == 1`. Return `None` if no result gives it.

In [ ]:
def period_from_counts(counts, t, a, N):
    """The order r read from measured results with continued fractions, or None."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete period_from_counts() first.")

In [ ]:
for key in sorted(counts):
    m = int(key, 2)
    f = Fraction(m, 2 ** t).limit_denominator(15)
    print(f"m = {m:3d}:  {f}  denominator {f.denominator}   7^{f.denominator} mod 15 = {pow(7, f.denominator, 15)}")
r = period_from_counts(counts, t, 7, 15)
print("order found:", r, "   factors:", factors_from_order(7, r, 15))

## Step 7: factor 15 for every allowed a

The cell runs the whole algorithm, quantum order finding followed by the classical steps, for each a. With a = 4 or 11 the order is 2; with the others it is 4. Every a here gives the factors 3 and 5.

In [ ]:
for a in (2, 4, 7, 8, 11, 13):
    c = sim.run(order_finding(a, 8), shots=200).result().get_counts()
    r = period_from_counts(c, 8, a, 15)
    print(f"a = {a:2d}: {len(c)} different results,  order r = {r},  factors {factors_from_order(a, r, 15)}")

## Step 8: your personal check

Open the **Module 3 lab check** in Canvas. Question 1 shows your own A (one of 2, 4, 7, 8, 11, 13) and seed SEED. Type them below and run the next two cells. The check cell tests `order()`, `factors_from_order()`, `order_finding()` (every a, 3 to 6 counting qubits) and `period_from_counts()`. Only if every test passes does it print your **verification value**: the number of shots, out of 4,000, in which the course's reference order-finding circuit for A with 8 counting qubits gives a result whose continued fraction has the true order as its denominator, run with your seed. Type it into Canvas.

In [ ]:
A = 0         # your number from Canvas, for example 7
SEED = 0      # your seed from Canvas, for example 512

In [ ]:
import math
from fractions import Fraction

import numpy as np
from qsim import AerSimulator, QuantumCircuit, Statevector

A_VALUES = (2, 4, 7, 8, 11, 13)


# ---------------------------------------------------------------- prepared parts (also given to learners)
def c_amod15(a, power):
    """Prepared: controlled multiplication by a^power mod 15, on qubits [control, x0, x1, x2, x3].
    Hard-coded for N = 15 and a in (2, 4, 7, 8, 11, 13): swaps and X gates that permute the 15 residues."""
    if a not in A_VALUES:
        raise ValueError("c_amod15 works only for a = 2, 4, 7, 8, 11 or 13")
    U = QuantumCircuit(5)
    for _ in range(power):
        if a in (2, 13):
            U.cswap(0, 3, 4); U.cswap(0, 2, 3); U.cswap(0, 1, 2)
        if a in (7, 8):
            U.cswap(0, 1, 2); U.cswap(0, 2, 3); U.cswap(0, 3, 4)
        if a in (4, 11):
            U.cswap(0, 2, 4); U.cswap(0, 1, 3)
        if a in (7, 11, 13):
            for q in range(1, 5):
                U.cx(0, q)
    return U


def qft_dagger(t):
    """Prepared: the inverse QFT on t qubits (the inverse of Module 1's qft(t), gates in reverse order)."""
    qc = QuantumCircuit(t)
    for i in range(t // 2):
        qc.swap(i, t - 1 - i)
    for j in range(t):
        for k in range(j):
            qc.cp(-math.pi / 2 ** (j - k), k, j)
        qc.h(j)
    return qc


# ---------------------------------------------------------------- reference solutions
def reference_order(a, N):
    r, x = 1, a % N
    while x != 1:
        x = (x * a) % N
        r += 1
    return r


def reference_factors(a, r, N):
    if r % 2:
        return None
    h = pow(a, r // 2, N)
    if h == N - 1:
        return None
    return tuple(sorted((math.gcd(h - 1, N), math.gcd(h + 1, N))))


def reference_order_finding(a, t, x_reg=True, h=True, inverse=True):
    qc = QuantumCircuit(t + 4, t)
    if x_reg:
        qc.x(t)
    if h:
        qc.h(list(range(t)))
    for k in range(t):
        qc.compose(c_amod15(a, 2 ** k), qubits=[k] + list(range(t, t + 4)), inplace=True)
    if inverse:
        qc.compose(qft_dagger(t), qubits=list(range(t)), inplace=True)
    qc.measure(list(range(t)), list(range(t)))
    return qc


def reference_period(counts, t, a, N):
    for key in sorted(counts, key=counts.get, reverse=True):
        r = Fraction(int(key, 2), 2 ** t).limit_denominator(N).denominator
        if pow(a, r, N) == 1:
            return r
    return None


def counting_distribution(qc, t):
    """Exact probabilities of the counting-register results (final measurements removed)."""
    c = qc.copy()
    c.remove_final_measurements(inplace=True)
    p = Statevector(c).probabilities(list(range(t)))
    return {format(m, f"0{t}b"): float(v) for m, v in enumerate(p) if v > 1e-12}


def personal_value(a, seed):
    a, t = int(a), 8
    r = reference_order(a, 15)
    counts = AerSimulator(seed_simulator=int(seed)).run(reference_order_finding(a, t), shots=4000).result().get_counts()
    return int(sum(c for k, c in counts.items()
                   if Fraction(int(k, 2), 2 ** t).limit_denominator(15).denominator == r))


# ---------------------------------------------------------------- tests
ORDER_CASES = [(7, 15), (2, 15), (4, 15), (11, 15), (13, 15), (2, 21), (5, 21), (10, 21), (3, 7), (2, 35)]


def _test_order(order):
    for a, N in ORDER_CASES:
        try:
            r = order(a, N)
        except NotImplementedError:
            return False, ["order() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"order({a}, {N}) raised {type(e).__name__}: {e}"]
        want = reference_order(a, N)
        if r != want:
            hint = ""
            if isinstance(r, int) and r > 0 and pow(a, r, N) == 1:
                hint = f" {a}^{r} mod {N} is 1, but a smaller power already gives 1: return the smallest r."
            elif r == want - 1:
                hint = " Off by one: count the powers a^1, a^2, ... and return the first r with a^r mod N = 1."
            return False, [f"order({a}, {N}) returned {r!r}; it should be {want}.{hint}"]
    return True, [f"order(): passed ({len(ORDER_CASES)} cases, N = 7 to 35)."]


FACTOR_CASES = [(7, 4, 15), (2, 4, 15), (4, 2, 15), (11, 2, 15), (13, 4, 15), (2, 6, 21), (10, 6, 21), (5, 6, 21),
                (14, 2, 15), (2, 3, 7)]


def _norm(x):
    if x is None:
        return None
    try:
        return tuple(sorted(int(v) for v in x))
    except Exception:  # noqa: BLE001
        return ("unreadable", repr(x))


def _test_factors(factors_from_order):
    for a, r, N in FACTOR_CASES:
        try:
            got = factors_from_order(a, r, N)
        except NotImplementedError:
            return False, ["factors_from_order() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"factors_from_order({a}, {r}, {N}) raised {type(e).__name__}: {e}"]
        want = reference_factors(a, r, N)
        if _norm(got) != want:
            if want is None and r % 2:
                hint = " r is odd, so a^(r/2) is not a whole power: return None."
            elif want is None:
                hint = f" Here {a}^{r // 2} mod {N} = {N - 1}, so both gcds are trivial (1 and {N}): return None."
            else:
                hint = " Use math.gcd(h - 1, N) and math.gcd(h + 1, N) with h = pow(a, r // 2, N), and return them sorted."
            return False, [f"factors_from_order({a}, {r}, {N}) returned {got!r}; it should be {want!r}.{hint}"]
    return True, [f"factors_from_order(): passed ({len(FACTOR_CASES)} cases, including odd r and a^(r/2) = -1 mod N)."]


OF_CASES = [(7, 3), (2, 4), (4, 4), (11, 3), (13, 5), (8, 4), (7, 6)]


def _test_order_finding(order_finding):
    for a, t in OF_CASES:
        try:
            qc = order_finding(a, t)
        except NotImplementedError:
            return False, ["order_finding() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"order_finding({a}, {t}) raised {type(e).__name__}: {e}"]
        if not isinstance(qc, QuantumCircuit) or qc.num_qubits != t + 4 or qc.num_clbits != t:
            return False, [f"order_finding({a}, {t}) should return a circuit with {t + 4} qubits and {t} classical bits."]
        meas = [i for i in qc.data if i.name == "measure"]
        if sorted(i.qubits[0] for i in meas) != list(range(t)) or any(i.qubits[0] != i.clbits[0] for i in meas):
            return False, ["Measure each counting qubit k (0 to t - 1) into classical bit k, and do not measure the register."]
        got = counting_distribution(qc, t)
        ref = counting_distribution(reference_order_finding(a, t), t)
        keys = set(got) | set(ref)
        if 0.5 * sum(abs(got.get(k, 0) - ref.get(k, 0)) for k in keys) > 1e-6:
            hint = ""
            if len(got) == 2 ** t and max(got.values()) - min(got.values()) < 1e-9:
                hint = (" Every result is equally likely: check that every counting qubit gets an H first and that the "
                        "inverse QFT, qft_dagger(t), is applied to the counting qubits before measuring.")
            for kw, h in ((dict(x_reg=False), " Start the register in |1>: an X on qubit t."),
                          (dict(h=False), " Put an H on every counting qubit first."),
                          (dict(inverse=False), " Apply the inverse QFT, qft_dagger(t), to the counting qubits before measuring.")):
                alt = counting_distribution(reference_order_finding(a, t, **kw), t)
                if not hint and 0.5 * sum(abs(got.get(k, 0) - alt.get(k, 0)) for k in set(got) | set(alt)) < 1e-6:
                    hint = h
                    break
            return False, [f"For a = {a} with {t} counting qubits, the results do not match the expected distribution.{hint} "
                           f"Counting qubit k controls c_amod15(a, 2**k), composed on qubits [k] + list(range(t, t + 4))."]
    return True, [f"order_finding(): passed ({len(OF_CASES)} cases, every a, 3 to 6 counting qubits; each distribution exact)."]


def _peaks(a, t):
    r = reference_order(a, 15)
    return {format(s * 2 ** t // r, f"0{t}b"): 1000 // r for s in range(r)}


PERIOD_CASES = [
    (_peaks(7, 8), 8, 7, 15),
    (_peaks(4, 8), 8, 4, 15),
    (_peaks(13, 4), 4, 13, 15),
    ({"10000000": 600, "00000000": 400, "01000000": 3}, 8, 2, 15),   # the 1/4 peak is rare but present
    ({"00101011": 300, "01010101": 310, "00000000": 290, "10101011": 50, "11010101": 40}, 8, 2, 21),  # near s/6
    ({"00000000": 500, "10000000": 500}, 8, 7, 15),                  # no result gives the order 4
]
PERIOD_WANT = [4, 2, 4, 4, 6, None]


def _test_period(period_from_counts):
    for (counts, t, a, N), want in zip(PERIOD_CASES, PERIOD_WANT):
        try:
            got = period_from_counts(dict(counts), t, a, N)
        except NotImplementedError:
            return False, ["period_from_counts() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"period_from_counts() raised {type(e).__name__}: {e} (counts with {len(counts)} results, t = {t}, a = {a}, N = {N})."]
        if got != want:
            if want is None:
                hint = (" No result in these counts gives a valid order (7^1 and 7^2 are not 1 mod 15), so return None "
                        "instead of a wrong r.")
            else:
                hint = (" For each result m, take Fraction(m, 2**t).limit_denominator(N).denominator and keep the first "
                        "candidate r with pow(a, r, N) == 1.")
            return False, [f"period_from_counts() returned {got!r} for a = {a}, N = {N} (results {sorted(counts)[:4]}...); it should be {want!r}.{hint}"]
    return True, [f"period_from_counts(): passed ({len(PERIOD_CASES)} cases, including N = 21 and counts that hold no valid order)."]


def check_l2_module3(order, factors_from_order, order_finding, period_from_counts, A, SEED):
    try:
        a, seed = int(A), int(SEED)
        if float(A) != a or float(SEED) != seed:
            raise ValueError
    except (TypeError, ValueError):
        return False, ["A and SEED must be the whole numbers shown in your Canvas lab check."], None
    if a not in A_VALUES or not (100 <= seed <= 999):
        return False, ["A must be one of 2, 4, 7, 8, 11, 13 and SEED between 100 and 999. Copy them again from the lab check."], None
    msgs, ok = [], True
    for test, fn in ((_test_order, order), (_test_factors, factors_from_order),
                     (_test_order_finding, order_finding), (_test_period, period_from_counts)):
        good, m = test(fn)
        msgs += m
        ok = ok and good
    if not ok:
        return False, msgs, None
    return True, msgs, personal_value(a, seed)


passed, messages, value = check_l2_module3(order, factors_from_order, order_finding, period_from_counts, A, SEED)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## What you should notice

- Factoring reduces to order finding: an even order r with a^(r/2) ≠ −1 mod N gives the factors gcd(a^(r/2) ± 1, N).
- Order finding is phase estimation of "multiply by a mod N", started in |1⟩; each shot gives m/2ᵗ ≈ s/r for a random s.
- Continued fractions turn m/2ᵗ into s/r; when s shares a factor with r, the denominator is too small, so check it and use another shot.
- The modular multiplication here was hard-coded for N = 15. For a real N it must be built from arithmetic circuits, and that, with error correction, is why breaking RSA-2048 needs a large fault-tolerant quantum computer that does not exist today.